[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Artificial Neural Networks

We build a neural network **from scratch in NumPy** — every forward pass, every gradient, every update written by hand — and train it until a spiral no line could separate falls to a few dozen lines of code. After this, [PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) will feel like a convenience, not a mystery.

## 0. Introduction

An artificial neuron computes $\sigma(\mathbf{w}^T\mathbf{x} + b)$: a weighted vote followed by a nonlinear squeeze. One neuron draws a single line through the data. The story of this workshop: *stack* votes into layers and the network bends that line into any boundary you need.

## 1. Pre-requisites

- [Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb) — NumPy fluency.
- The chain rule from calculus — backpropagation *is* the chain rule, organized.
- [Adaptive Filtering: APA](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) is a helpful cousin: LMS is literally training a single linear neuron.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)

---
### 🕐 Session 1 of 3 — *From Neuron to Network* (~35 min)
**Goal:** understand the perceptron, why nonlinearity is essential, and why depth buys expressiveness.
**Feeds into:** Session 2 (backpropagation).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: From Neuron to Network</b></summary>

**Timing (~35 min).** 8 min the neuron as a weighted vote · 10 min the spiral and why a line fails · 12 min why nonlinearity is non-negotiable · 5 min activations.

**Open with the one-line description of a neuron and make it feel familiar rather than biological.** $\sigma(\mathbf{w}^T\mathbf{x} + b)$ is a weighted vote followed by a squeeze. The vote is an inner product — Session 1 of [Linear Algebra](../../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb), the same operation as a projection or a filter tap. Resist the brain analogy; it buys nothing here and misleads later. A neuron draws **one line** through the data, and everything interesting comes from what happens when you stack them.

**Show the spiral before any theory and let the room try to beat it.** Ask for a line that separates the two classes. There isn't one, and after thirty seconds of trying, the students own the problem rather than being told about it. Worth naming that this is not a hard dataset by modern standards — it is 600 points in two dimensions — and a single neuron is still helpless against it.

**Then deliver the session's central argument, which is the one thing students must not leave without.** Stack linear layers and they **collapse**: $W_2(W_1x) = (W_2W_1)x$, and a product of matrices is just another matrix. A hundred linear layers is one line in disguise. Do this on the board; it is one line of algebra and it settles the question. Depth without nonlinearity buys **nothing at all** — not "less", literally nothing, since the function class is identical.

**Give the geometric picture too, because it explains what depth actually does.** Each ReLU unit contributes one *fold* of the input space — a hyperplane on one side of which the unit is silent. With 32 units the plane is creased into many regions, and layers of folds crumple it until the spirals can be pulled apart by a single line in the final layer. The last layer is always a linear classifier; the hidden layers exist to hand it a space where a line suffices. That framing makes the boundary plot in Session 3 legible in advance.

**On the activation choices, give the reason rather than the convention.** ReLU is cheap (a comparison), and its gradient is exactly 1 for active units — so it does not shrink the signal as it flows backward, unlike sigmoid, whose derivative peaks at 0.25 and decays to nothing at either extreme. Stack ten sigmoids and gradients get multiplied by $\le 0.25$ ten times: $10^{-6}$, and the early layers stop learning. **That is the vanishing-gradient problem, and it is why deep networks became practical only after ReLU.** Sigmoid survives on the *output*, where we want a number in $(0,1)$ to read as a probability and where there is nothing below it to starve.

**If the room asks about ReLU's dead units, answer honestly.** A unit whose input is negative for every training example has zero gradient forever and never recovers — "dying ReLU". It is a real failure mode, mitigated by Leaky ReLU or GELU, and it is the price paid for the flat region that makes the gradient well-behaved elsewhere.

**Preview the experiment at the end of Session 3 now, so it lands as a prediction rather than a demo.** Setting `A1 = Z1` removes the nonlinearity, and the boundary should collapse to a straight line. Ask the room to commit to that prediction before they see it; the claim made here is falsifiable in three keystrokes, and a session that ends by testing its own opening claim is worth more than one that asserts it twice.
</details>

## 2. Theory: Neurons, Layers, Nonlinearity

### 2.1. The Dataset That Defeats a Line

Two interleaved spirals. No single weighted vote — no *line* — can separate them.

In [2]:
def spirals(n_per_class=300, noise=0.25):
    t = np.linspace(0.5, 3 * np.pi, n_per_class)
    X, y = [], []
    for cls, phase in [(0, 0.0), (1, np.pi)]:
        x1 = t * np.cos(t + phase) + noise * rng.standard_normal(n_per_class)
        x2 = t * np.sin(t + phase) + noise * rng.standard_normal(n_per_class)
        X.append(np.stack([x1, x2], axis=1)); y.append(np.full(n_per_class, cls))
    X = np.concatenate(X); y = np.concatenate(y)
    X = (X - X.mean(0)) / X.std(0)                # standardize
    return X, y

X, y = spirals()
plt.figure(figsize=(4.5, 4.5))
plt.scatter(*X[y == 0].T, s=8, label="class 0")
plt.scatter(*X[y == 1].T, s=8, label="class 1")
plt.legend(); plt.title("Try separating THIS with a line")
plt.axis("equal"); plt.tight_layout(); plt.show()

/tmp/ipykernel_1840136/294026007.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.axis("equal"); plt.tight_layout(); plt.show()


**What just happened.** Two interleaved arms, 300 points each, standardized to zero mean and unit variance. Take a moment and actually try to draw a separating line — any line. Whatever you choose, it cuts through both spirals, because each arm wraps around the origin through more than a full turn and revisits every direction.

**That failure is structural, not a matter of finding a better line.** A single neuron computes $\sigma(\mathbf{w}^Tx + b)$, and the set where it flips its vote is $\mathbf{w}^Tx + b = 0$ — a straight line, full stop. There are only three free numbers and none of them can produce a curve. So a perceptron on this data is capped at roughly 50% accuracy: chance.

**Note that the difficulty has nothing to do with size.** 600 points, two dimensions, no missing values, mild noise, perfectly balanced classes. By any modern measure this is a tiny, clean dataset. It is hard for exactly one reason — **the decision boundary is not linear** — and that single property is what motivates every layer added from here on.

**Two details in the generator are worth pointing at, since students will reuse this code.** The two classes are the *same* spiral with a $\pi$ phase offset, so they are perfectly interleaved by construction rather than by luck; and the final standardization is not cosmetic. He initialization (`sqrt(2/fan_in)` in the next cell) assumes inputs of roughly unit scale, and the raw spiral runs out to radius $3\pi \approx 9.4$. Feed the unstandardized data in and the first layer saturates immediately. **Standardization is a precondition for the initialization scheme, not a nicety.**

**Keep this picture in view for the rest of the workshop, because it is the yardstick.** In Session 3 the same plane gets painted by a trained network's predictions, and the boundary follows both arms around. Nothing changes about the data between here and there — what changes is that 32 hidden units have folded the space until a line in the final layer is enough.

💡 **Intuition.** Why nonlinearity is non-negotiable: stacking *linear* layers collapses — a matrix times a matrix is just another matrix, so a 100-layer linear network is one line in disguise. The activation function between layers breaks that collapse. With it, each hidden neuron contributes one *fold* of the input space; layers of folds crumple the plane until the spirals become linearly separable in the last layer.

### 2.2. Activations

We'll use **ReLU** ($\max(0, u)$) in hidden layers — cheap, and its gradient doesn't vanish for active units — and a **sigmoid** on the output to read the result as a probability.

In [3]:
u = np.linspace(-4, 4, 200)
relu = np.maximum(0, u)
sigmoid = 1 / (1 + np.exp(-u))

fig, axes = plt.subplots(1, 2, figsize=(8, 2.4))
axes[0].plot(u, relu); axes[0].set_title("ReLU: max(0, u)")
axes[1].plot(u, sigmoid); axes[1].set_title("sigmoid: 1/(1+e⁻ᵘ)")
for ax in axes: ax.grid(True)
plt.tight_layout(); plt.show()

/tmp/ipykernel_1840136/1137002607.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two very different shapes. ReLU is a hinge — flat zero to the left, slope exactly 1 to the right. The sigmoid is a smooth S, squashing all of $\mathbb{R}$ into $(0,1)$ and flattening out at both ends.

**Read them by their *derivatives*, because that is what backprop multiplies.** ReLU's derivative is 1 wherever the unit is active and 0 where it is not — so an active path passes the gradient through **unattenuated**. The sigmoid's derivative is $\sigma(1-\sigma)$, which peaks at **0.25** at the origin and falls toward zero at both extremes. That single number is the reason these two functions have different jobs.

**Multiply it out and the historical argument writes itself.** Stack ten sigmoid layers and the backward pass multiplies at most $0.25$ ten times: $0.25^{10} \approx 10^{-6}$. The early layers receive a gradient a million times smaller than the late ones and effectively stop learning. **That is the vanishing-gradient problem**, and it is why networks deeper than a few layers were impractical for decades. ReLU's slope of 1 removes the attenuation entirely, and deep learning became trainable.

**Which is why the two appear in different places, deliberately.** ReLU sits in the hidden layers, where the only requirement is a nonlinearity that does not strangle the gradient. Sigmoid sits on the *output*, where we genuinely want a number in $(0,1)$ to read as $P(\text{class }1)$ — and where saturation costs nothing, because there is no layer beneath it to starve.

**Note the price ReLU pays for that flat region.** A unit whose pre-activation is negative for **every** training example receives exactly zero gradient, forever, and can never recover — the "dying ReLU". It is a real failure mode, not a footnote; Leaky ReLU and GELU exist to give the negative side a small nonzero slope precisely to avoid it. Flatness is what makes the gradient clean on one side and fatal on the other.

**And one thing neither plot shows, which is the actual point of nonlinearity.** These are not chosen for their curves but for being *not linear*. Any of ReLU, sigmoid, tanh, GELU breaks the collapse $W_2(W_1x) = (W_2W_1)x$ and makes depth mean something. The differences between them are matters of gradient flow and cost — the requirement they all satisfy is the one from the previous cell, and it is the only requirement that is non-negotiable.

---
### 🕐 Session 2 of 3 — *Backpropagation* (~35 min)
**Goal:** derive the gradients as the chain rule on a computational graph — and implement them.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (training).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Backpropagation</b></summary>

**Timing (~35 min).** 10 min blame assignment as a concept · 12 min deriving the four gradients · 8 min reading the implementation · 5 min the gradient check.

**Lead with the reframe, because "backpropagation" sounds like an algorithm and is really an accounting discipline.** The loss says the prediction was off by some amount. Walking backward, every operation answers one local question: *given how much my output was to blame, how much were my inputs and my weights to blame?* The answer is that operation's local derivative. Multiplying local blames along a path is the chain rule. **Backprop is the chain rule with the bookkeeping done once per node instead of once per weight** — that is the entire idea, and it is worth stating before any notation appears.

**Say why the bookkeeping matters, since students often ask why this needs a name.** Deriving $\partial L/\partial w$ separately for each weight repeats enormous amounts of shared work; a network with $10^6$ parameters would need $10^6$ derivations. Backprop computes all of them in one backward sweep costing about the same as one forward pass. That efficiency is the reason the method has a name and the reason deep learning is computationally possible.

**Derive $\delta_2 = \hat y - y$ at the board rather than quoting it, because the cancellation is beautiful and load-bearing.** Cross-entropy contributes $\partial L/\partial\hat y = \frac{\hat y - y}{\hat y(1-\hat y)}$; the sigmoid contributes $\partial\hat y/\partial z = \hat y(1-\hat y)$. The denominators cancel exactly and leave **predicted minus true**. Then make the connection explicit: this is the same "error signal" as the LMS update in [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb), and the same $\hat y - y$ that appears in the normal equations of [Linear Algebra](../../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) Session 2. Three workshops, one quantity.

**Point out that the cancellation is engineered, not lucky.** Pairing sigmoid with cross-entropy is a deliberate choice: use squared error with a sigmoid instead and you keep the $\hat y(1-\hat y)$ factor, which vanishes exactly when the network is confidently wrong — the case you most need a large gradient for. **The loss and the output activation are chosen as a pair**, and this is why softmax pairs with categorical cross-entropy everywhere in practice.

**Read the ReLU mask aloud as a sentence: neurons that were off take no blame.** $\mathbf{1}[Z_1 > 0]$ is not a mathematical formality — it says that a unit which contributed nothing to the prediction is not responsible for the error, so its weights are not updated on that example. Students find the code line `d1 = (d2 @ p["W2"].T) * (Z1 > 0)` much easier to trust once they have heard it in words.

**Walk the shapes on the board if the room is at all shaky on matrix calculus.** $X$ is $n \times 2$, $W_1$ is $2 \times 16$, $\delta_1$ is $n \times 16$, so $X^T\delta_1$ is $2 \times 16$ — the same shape as $W_1$, as it must be. **Shape-checking catches most backprop bugs before they run**, and it is a habit worth teaching explicitly. Note also the $1/m$: gradients are averaged over the batch so the learning rate does not have to be retuned when the batch size changes.

**Then treat the gradient check as the session's professional lesson, not a demo.** A wrong gradient does not raise an exception. It trains slowly, plateaus, or converges somewhere odd, and it can burn days. Central differences cost five lines and milliseconds. Frame the $\varepsilon$ choice honestly: too large and truncation dominates, too small and floating-point cancellation destroys the numerator; $10^{-5}$ balances them. Every autodiff framework ships a `gradcheck` for exactly this reason, and [Linear Algebra](../../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) Session 5 made the same point about hand-derived gradients.
</details>

## 3. Theory: Backpropagation

💡 **Intuition.** Backprop is *blame assignment*. The loss says "prediction off by this much." Walking backward through the network, each operation answers one local question: *given how much my output was to blame, how much were my inputs and weights to blame?* — that answer is its local derivative. Multiplying local blames along the path is exactly the chain rule; "backprop" is just doing it once per node, back-to-front, instead of re-deriving a formula per weight.

### 3.1. The Two-Layer Network

Forward pass, batch $X$ ($n \times 2$):

$$Z_1 = X W_1 + \mathbf{b}_1 \quad A_1 = \mathrm{ReLU}(Z_1) \quad Z_2 = A_1 W_2 + \mathbf{b}_2 \quad \hat{y} = \sigma(Z_2)$$

Loss — binary cross-entropy: $\;L = -\frac{1}{n}\sum y\log\hat{y} + (1-y)\log(1-\hat{y})$

### 3.2. The Gradients

The famous simplification: for sigmoid + cross-entropy, the output blame collapses to $\delta_2 = \hat{y} - y$ (predicted minus true — the *error*, again!). Then

$$\nabla W_2 = \tfrac{1}{n} A_1^T \delta_2 \qquad \delta_1 = (\delta_2 W_2^T) \odot \mathbf{1}[Z_1 > 0] \qquad \nabla W_1 = \tfrac{1}{n} X^T \delta_1$$

The ReLU mask $\mathbf{1}[Z_1>0]$ says: neurons that were off take no blame.

In [4]:
def init(sizes):
    params = {}
    for i, (fan_in, fan_out) in enumerate(zip(sizes[:-1], sizes[1:]), 1):
        params[f"W{i}"] = rng.standard_normal((fan_in, fan_out)) * np.sqrt(2 / fan_in)  # He init
        params[f"b{i}"] = np.zeros(fan_out)
    return params

def forward(p, X):
    Z1 = X @ p["W1"] + p["b1"]
    A1 = np.maximum(0, Z1)
    Z2 = A1 @ p["W2"] + p["b2"]
    yhat = 1 / (1 + np.exp(-Z2.ravel()))
    return yhat, (X, Z1, A1)

def backward(p, cache, yhat, y):
    X, Z1, A1 = cache
    m = len(y)
    d2 = (yhat - y).reshape(-1, 1)                    # output blame
    grads = {"W2": A1.T @ d2 / m, "b2": d2.mean(0)}
    d1 = (d2 @ p["W2"].T) * (Z1 > 0)                  # ReLU mask
    grads["W1"] = X.T @ d1 / m
    grads["b1"] = d1.mean(0)
    return grads

### 3.3. Trust, but Verify: the Gradient Check

The classic backprop bug is a silently wrong gradient. The antidote: compare against a finite difference $\frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}$ on a few random weights.

In [5]:
def loss_of(p, X, y):
    yhat, _ = forward(p, X)
    eps = 1e-12
    return -np.mean(y * np.log(yhat + eps) + (1 - y) * np.log(1 - yhat + eps))

p = init([2, 16, 1])
yhat, cache = forward(p, X)
grads = backward(p, cache, yhat, y)

eps = 1e-5
for name, idx in [("W1", (0, 3)), ("W2", (7, 0)), ("b1", (2,))]:
    p[name][idx] += eps;  lp = loss_of(p, X, y)
    p[name][idx] -= 2 * eps; lm = loss_of(p, X, y)
    p[name][idx] += eps
    numeric = (lp - lm) / (2 * eps)
    analytic = grads[name][idx]
    print(f"{name}{idx}: analytic {analytic:+.6f}  numeric {numeric:+.6f}")
    assert abs(numeric - analytic) < 1e-6

W1(0, 3): analytic -0.003470  numeric -0.003470
W2(7, 0): analytic -0.055097  numeric -0.055097
b1(2,): analytic +0.069507  numeric +0.069507


**What just happened.** Three weights sampled from three different tensors, and analytic and numeric gradients agree to all six printed digits:

| parameter | analytic | numeric |
|---|---|---|
| `W1[0,3]` | −0.003470 | −0.003470 |
| `W2[7,0]` | −0.055097 | −0.055097 |
| `b1[2]` | +0.069507 | +0.069507 |

And the `assert` demands agreement below $10^{-6}$, so this is a **test that can fail**, not a printout to admire. The hand-derived backward pass is correct.

**Note that the three checks are not redundant — each exercises a different part of the derivation.** `W2` tests only the output layer and the $\delta_2 = \hat y - y$ collapse. `W1` tests the full path, including the ReLU mask and the $\delta_2W_2^T$ propagation — the step where sign errors and missing transposes actually live. `b1` tests the bias reduction, which is the one place a `sum` and a `mean` are easy to confuse. A check that only sampled `W2` would pass with a badly broken `W1`.

**Why this matters more than it appears: a wrong gradient does not crash.** It produces a network that trains a little slowly, plateaus at a mediocre loss, or converges somewhere odd — symptoms indistinguishable from "the model needs more capacity" or "the learning rate is off". Practitioners have lost days to a transposed matrix that ran perfectly. **The failure mode of backprop is silence**, and that is precisely why an explicit check earns its five lines.

**The $\varepsilon = 10^{-5}$ choice is a real numerical decision, not a magic number.** The central difference has truncation error $O(\varepsilon^2)$ and cancellation error $O(\varepsilon_{\text{mach}}/\varepsilon)$; the two balance near $\sqrt[3]{\varepsilon_{\text{mach}}} \approx 10^{-5}$ in double precision. Try $10^{-12}$ and the numerator becomes the difference of two nearly identical doubles — the check starts reporting large errors for a *correct* gradient, and the natural but wrong conclusion is that the gradient is broken.

**One caveat on ReLU that is worth knowing before students hit it.** ReLU is not differentiable at zero, so if a perturbation pushes some $Z_1$ entry across the hinge, the finite difference and the analytic gradient will genuinely disagree — not a bug in either, but a real kink in the function. With 600 examples and random weights it is unlikely enough not to matter here, and it is why gradient checks on ReLU networks are sometimes run with a smooth activation substituted in.

**Finally, note that this is the same discipline the frameworks institutionalise.** `torch.autograd.gradcheck` does exactly this against autograd's output, and it exists because even the people who wrote the autodiff engine do not trust hand-written backward methods without testing them. Writing the check yourself once is what makes that habit portable.

---
### 🕐 Session 3 of 3 — *Training the Network* (~40 min)
**Goal:** run the full training loop on the spiral, visualize the learned boundary, then meet PyTorch.
**Builds on:** Session 2. &nbsp; **Feeds into:** [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Training the Network</b></summary>

**Timing (~40 min).** 10 min the four-beat loop · 10 min the loss curve · 10 min the decision boundary · 10 min the three experiments.

**Name the loop before running it, because everything in deep learning is this loop.** Predict (forward), grade (loss), diagnose (backward), nudge (update). Four beats, five lines of Python, and every model in the curriculum — CNNs, transformers, diffusion — runs exactly this. Frameworks change *who writes* the diagnose step, not what the step is. Students who leave with the four beats internalised can read any training script.

**Have the room predict the final accuracy before running, then let 100% land.** It usually surprises people that 32 hidden units and 3000 plain gradient steps — no momentum, no Adam, no batching, no regularisation — perfectly separate a spiral. That is worth a moment. It is also worth immediately puncturing: **this is training accuracy on 600 points with no held-out set**, so it measures fitting, not learning. See the debrief below for the honest version; do not let the number stand unqualified.

**Read the loss curve for its *shape*, not its endpoint.** A long flat plateau at the start (the network is still finding a useful representation), then a steep drop, then a slow grind. The plateau is the interesting part: gradient descent on a non-convex surface can spend hundreds of epochs going almost nowhere before the geometry improves. Students who expect monotone rapid progress conclude their code is broken during exactly this phase.

**The boundary plot is the session's payoff — spend real time on it.** Ask what the network should have learned, then show the painted plane. The boundary spirals. Then push further: ask *how* a stack of straight-line units produced a curve. The answer is Session 1's folding picture — 32 ReLU hinges partition the plane into many polygonal regions, the final layer draws one line in the folded space, and unfolding turns that line into a spiral. **The boundary is piecewise linear**, and zooming in far enough would show the facets. That is a genuinely useful thing to know about ReLU networks.

**Run the "remove the ReLU" experiment live if there is any time at all.** Setting `A1 = Z1` and re-running collapses the boundary to a straight line and the accuracy to about 50%. It takes thirty seconds and it *proves* Session 1's central claim rather than restating it. Ask the room to predict the outcome first; a falsifiable prediction they made themselves is worth more than any amount of assertion.

**The width experiment teaches capacity, and is worth framing as a tradeoff rather than a dial.** Width 4 underfits — the boundary is too coarse to follow both arms. Width 32 is about right. Width 256 fits the noise: the boundary grows fingers reaching for individual points. Connect this to [Linear Algebra](../../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) Session 2, where projecting onto a larger subspace captured more noise — the same bias–variance curve, now in parameter count.

**The learning-rate experiment is the cheapest lesson in the workshop.** $\eta = 5.0$ diverges to NaN in a few epochs. Tie it back to [Linear Algebra](../../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) Session 3: the stability bound $\mu < 2/\lambda_{\max}$ from LMS is the same phenomenon, with $\lambda_{\max}$ now being the largest curvature of a non-convex loss. Divergence is not mysterious — it is the step size exceeding the curvature the surface can absorb.

**Close by cashing in the workshop's promise.** Everything deep learning does was in this notebook: forward pass, loss, chain-rule blame assignment, gradient step. [PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) adds autograd, GPU tensors, and a layer library — conveniences on top of *this* loop, not a different idea. Students who have written the backward pass by hand once read framework code as recognition rather than magic, which is the entire reason this workshop exists in NumPy.
</details>

## 4. Application: Train on the Spiral

In [6]:
p = init([2, 32, 1])
lr, losses = 0.5, []

for epoch in range(3000):
    yhat, cache = forward(p, X)
    grads = backward(p, cache, yhat, y)
    for k in p:
        p[k] -= lr * grads[k]
    if epoch % 50 == 0:
        losses.append(loss_of(p, X, y))

acc = np.mean((forward(p, X)[0] > 0.5) == y)
print(f"final training accuracy: {acc:.1%}")

plt.figure(figsize=(7, 2.5))
plt.plot(np.arange(len(losses)) * 50, losses)
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss"); plt.grid(True)
plt.title("The four-beat loop: predict, grade, diagnose, nudge")
plt.tight_layout(); plt.show()

final training accuracy: 100.0%


/tmp/ipykernel_1840136/3456064341.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** **100.0% training accuracy** on the dataset that defeated every line, and the loss curve shows how: a slow start, then a steep collapse, then a long grind toward zero. Thirty-two hidden units and 3000 plain gradient steps — no momentum, no Adam, no mini-batching, no regularisation, no framework.

**Read the curve's shape rather than its endpoint, because the shape is the lesson.** The early plateau is not the code failing to work; it is gradient descent on a non-convex surface, where the network has not yet found a representation that makes the spiral tractable and the gradient is genuinely small. Then the drop, when useful folds appear. Then diminishing returns, as the remaining loss comes from points near the boundary that the network is merely growing more confident about. **Students who abandon a run during the plateau abandon runs that would have worked.**

**Now the honest caveat, because 100% is exactly the number that should make you suspicious.** This is **training** accuracy, on the same 600 points the network optimised against, with **no held-out set anywhere in this notebook**. It measures memorisation capacity, not generalisation. A network with 32 hidden units has $2 \times 32 + 32 + 32 + 1 = 129$ parameters against 600 examples — comfortably enough to fit, and comfortably enough to fit the noise too. The correct claim is "the network can represent and fit this boundary", not "the network learned the spiral."

**Test the distinction rather than arguing it.** Generate a second spiral dataset with the same code and a fresh seed, then evaluate. Accuracy will drop — modestly here, because the boundary genuinely is a spiral and the model found it, but it will drop. That gap between train and test is the only quantity that ever tells you whether a model learned anything, and every workshop downstream of this one takes it as the primary metric.

**Two implementation notes worth catching while the numbers are on screen.** The loss is recorded every 50 epochs, so the curve has 60 points, not 3000 — the visible smoothness is partly sampling. And this is **full-batch** gradient descent: every step uses all 600 points, which is why a learning rate as large as 0.5 is stable. Mini-batch SGD injects gradient noise and would need a smaller rate or a schedule, which is the regime every real training script operates in.

**Finally, note what was *not* needed.** No adaptive optimiser, no batch norm, no dropout, no learning-rate schedule, no early stopping. The four-beat loop — predict, grade, diagnose, nudge — with a fixed step size was sufficient. Everything else in the modern toolkit exists to make this loop work on problems where it otherwise would not; none of it is a different idea.

In [7]:
# Visualize what the network learned: paint every point of the plane by its prediction
g = np.linspace(-2.5, 2.5, 300)
GX, GY = np.meshgrid(g, g)
grid_pred = forward(p, np.stack([GX.ravel(), GY.ravel()], axis=1))[0].reshape(GX.shape)

plt.figure(figsize=(5, 4.5))
plt.contourf(GX, GY, grid_pred, levels=30, cmap="RdBu", alpha=0.7)
plt.colorbar(label="P(class 1)")
plt.scatter(*X[y == 0].T, s=6, c="darkred")
plt.scatter(*X[y == 1].T, s=6, c="navy")
plt.title("32 hidden ReLUs folded the plane around the spiral")
plt.axis("equal"); plt.tight_layout(); plt.show()

/tmp/ipykernel_1840136/1526725041.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.axis("equal"); plt.tight_layout(); plt.show()


**What just happened.** The whole plane is painted by the network's output, and the red/blue frontier **spirals** — following both arms around, threading between them, staying confident (deep colour) where data is dense and hedging toward 0.5 (pale) in the gaps. Compare it to the first plot in Session 1, where the challenge was to draw a separating line. This is that line, after 32 hidden units got hold of the plane.

**The key question to ask here: how did a stack of straight-line units produce a curve?** Every neuron computes $\mathbf{w}^Tx + b$ — nothing in the network can bend anything. The answer is Session 1's folding picture, made concrete. Each ReLU hinge splits the plane along a line, silent on one side; 32 hinges partition it into many polygonal regions, and within each region the network is exactly linear. The final layer draws **one straight line** in that folded coordinate system, and unfolding turns it into the spiral you see.

**Which means the boundary is piecewise linear, not smooth — and that is checkable.** Zoom in far enough and the spiral resolves into flat facets meeting at corners. A ReLU network can only ever produce a piecewise-linear function; its apparent smoothness is a matter of having more pieces than pixels. Worth stating plainly, because "neural networks learn smooth functions" is a common and false belief, and the piecewise structure explains real behaviour (adversarial examples live on those facets).

**Read the pale regions as the model's uncertainty, with a caveat.** Between the arms, where no training point falls, the output drifts toward 0.5 — the network hedges where it has no evidence. That is the desirable behaviour. But look at the corners of the plot, far outside the data: the network is *confident* out there, deeply coloured, on the basis of nothing at all. **Extrapolation confidence is not calibrated uncertainty.** A ReLU network extends its outermost linear pieces to infinity, so it will always have an opinion about regions it has never seen, and that opinion is an artifact of the folds rather than a claim about the world. This is exactly why [Uncertainty in ML](../Uncertainty_in_ML.ipynb) exists as its own workshop.

**Note also what the boundary reveals about capacity.** It hugs the arms without growing fingers toward individual points — 32 units is roughly the right size for this problem. Re-run with width 4 and the boundary is too coarse to follow both turns; with 256 it starts reaching for isolated noisy points, and the frontier acquires blobs and spurs that correspond to nothing real. Same bias–variance curve as the projection dimension in [Linear Algebra](../../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) Session 2, now measured in parameters.

**And the cheapest experiment in the workshop lives right here.** Set `A1 = Z1` in `forward`, retrain, and re-plot. The boundary collapses to a single straight line and accuracy falls to roughly 50%, because $W_2(W_1x)$ is just another matrix. Thirty seconds of work that turns Session 1's central claim from an assertion into a result you watched happen.

Experiments worth 5 minutes each (edit and re-run):

- Hidden width 4 vs 32 vs 256 — watch the boundary sharpen (and eventually overfit the noise).
- Remove the ReLU (`A1 = Z1`) — the boundary collapses to a line, *proving* Session 1's claim.
- Learning rate 5.0 — meet divergence in person.

## 5. Conclusion

Everything deep learning does was in this notebook: forward pass, loss, chain-rule blame assignment, gradient step. Frameworks add autograd (no hand-derived gradients), GPU tensors, and libraries of layers — conveniences on top of *this* loop.

---
## Where next

- [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) — the same network with autograd doing Session 2 for you.
- [Convolutional Neural Networks](../README.md#workshop-2--convolutional-neural-networks-available) — weight sharing turns layers into learned filter banks (bridging back to [DSP](../../Intro_DSP/README.md)).
- [Intro to Transformers](../../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — attention as data-dependent connectivity.